In [1]:
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import numpy as np

In [2]:
#pip install reportlab


In [3]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import datetime


In [4]:
from pathlib import Path
import numpy as np
import wfdb # type: ignore
import matplotlib.pyplot as plt
import pandas as pd
import csv
import pywt # type: ignore
from scipy.signal import hilbert
from scipy.signal import find_peaks
import ast

In [5]:
#variables fijas
SECONDS_TO_SHOW = 10

In [6]:
#Funciones 
def strip_wfdb_suffix(p: Path) -> Path:
    p = str(p).strip().strip('"').strip("'")
    
    p=Path(p)
    while p.suffix in (".dat", ".hea"):
        p = p.with_suffix("")
    return p

def leer_ecg(BASE):
    BASE = strip_wfdb_suffix(BASE)
    p_signal, fields = wfdb.rdsamp(str(BASE))
    fs = float(fields["fs"])
    lead_names = fields["sig_name"]
    n, L = p_signal.shape
    n_show = min(n, int(SECONDS_TO_SHOW * fs))
    x = np.arange(n_show) / fs  # eje tiempo en segundos
    sig = p_signal[:n_show, :]

    # Calcula un espaciado vertical cómodo (basado en mediana del pico-a-pico por derivación)
    ptp_per_lead = np.ptp(sig, axis=0)  # pico-a-pico por derivación
    spacing = 1.2 * np.median(ptp_per_lead) if np.median(ptp_per_lead) > 0 else 100.0

    return L,x,sig,spacing,BASE,n_show,lead_names,fs

def graficar_ecg(L,x,sig,spacing,BASE,n_show,lead_names,fs):#Dash no renderiza plt hay que usar plotly
    fig=go.Figure()
    for i in range(sig.shape[1]):  # número real de derivaciones

        fig.add_trace(
            go.Scatter(
                x=x,
                y=sig[:, i] + i * spacing,
                mode='lines',
                name=lead_names[i],
                line=dict(width=1)
            )
        )

    fig.update_layout(
        title=f"ECG 12 derivaciones — {Path(BASE).name} — Fs={fs:.0f} Hz",

        height=900,

        plot_bgcolor='white',
        paper_bgcolor='white',

        xaxis_title="Tiempo (s)",
        yaxis_title="Derivaciones",

        showlegend=False,

        margin=dict(l=60, r=30, t=60, b=40)
    )

    fig.update_yaxes(
        tickvals=[i * spacing for i in range(sig.shape[1])],
        ticktext=lead_names
    )

    return fig

In [7]:


app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([

    # ── CABECERA ──────────────────────────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            html.H2("🫀 Dashboard de Diagnóstico ECG",
                    style={'color': '#2C5F8A', 'marginBottom': '0'}),
            html.P("Análisis automático de electrocardiogramas mediante Machine Learning",
                   style={'color': '#666', 'marginTop': '4px'})
        ])
    ], style={'padding': '20px 0 10px 0', 'borderBottom': '2px solid #e0e0e0'}),

    html.Br(),

    # ── PANEL DE ENTRADA ──────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("Datos del paciente", className='card-title'),
            dbc.Row([
                dbc.Col([
                    dcc.Input(
                        id='input-ruta',
                        placeholder="Copia la ruta del fichero .hea de tu ECG",
                        style={
                            'width': '100%',
                            'height': '80px',
                            'lineHeight': '80px',
                            'borderWidth': '2px',
                            'borderStyle': 'dashed',
                            'borderRadius': '10px',
                            'textAlign': 'center',
                            'marginBottom': '10px'
                        },
                        multiple=False
                    )
                ], width=12)
                
            ]),
            html.Br(),
            dbc.Button("🔍 Analizar ECG", id='btn-analizar', color='primary', size='lg'),
        ])
    ], style={'marginBottom': '20px'}),

    # ── MENSAJES DE ERROR ─────────────────────────────────────────────────────
    html.Div(id='div-error'),

    # ── RESUMEN CLÍNICO ───────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("📊 Resumen clínico", className='card-title'),
            dbc.Row([
                dbc.Col(dbc.Card([dbc.CardBody([
                    html.H4("72", className='card-title text-center'),
                    html.P("bpm", className='text-center text-muted')
                ])], color='light'), width=2),
                dbc.Col(dbc.Card([dbc.CardBody([
                    html.H4("45°", className='card-title text-center'),
                    html.P("Eje eléctrico", className='text-center text-muted')
                ])], color='light'), width=2),
                dbc.Col(dbc.Card([dbc.CardBody([
                    html.H4("0.833 s", className='card-title text-center'),
                    html.P("Media RR", className='text-center text-muted')
                ])], color='light'), width=2),
                dbc.Col(dbc.Card([dbc.CardBody([
                    html.H4("0.045 s", className='card-title text-center'),
                    html.P("SDNN (HRV)", className='text-center text-muted')
                ])], color='light'), width=2),
                dbc.Col(dbc.Card([dbc.CardBody([
                    html.H4("11", className='card-title text-center'),
                    html.P("Latidos detectados", className='text-center text-muted')
                ])], color='light'), width=2),
            ])
        ])
    ], style={'marginBottom': '20px'}),

    # ── ECG COMPLETO ──────────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("ECG completo (12 derivaciones)", className='card-title'),
            dcc.Graph(id='grafico-ecg', figure=go.Figure(
                layout=dict(height=900, title='ECG — datos pendientes',
                            plot_bgcolor='white', paper_bgcolor='white')
            ))
        ])
    ], style={'marginBottom': '20px'}),

    # ── LATIDOS SEGMENTADOS ───────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("💓 Latidos segmentados", className='card-title'),
            dcc.Graph(id='grafico-latidos', figure=go.Figure(
                layout=dict(height=600, title='Latidos — datos pendientes',
                            plot_bgcolor='white', paper_bgcolor='white')
            ))
        ])
    ], style={'marginBottom': '20px'}),

    # ── DIAGNÓSTICO ───────────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("🏥 Diagnóstico", className='card-title'),
            html.Div([
                dbc.Badge("✅ NORM: 82%", color='success', className='me-2',
                          style={'fontSize': '14px'}),
                dbc.Badge("⚠️ HYP: 60%", color='warning', className='me-2',
                          style={'fontSize': '14px'}),
            ], style={'marginBottom': '15px'}),
            dcc.Graph(id='grafico-diagnostico', figure=go.Figure(
                go.Bar(
                    y=['NORM', 'STTC', 'CD', 'MI', 'HYP'],
                    x=[0.82,   0.45,   0.30,  0.15, 0.60],
                    orientation='h',
                    marker_color=['#1D9E75', '#378ADD', '#7F77DD', '#D85A30', '#BA7517'],
                    text=['82%', '45%', '30%', '15%', '60%'],
                    textposition='outside'
                ),
                layout=dict(
                    height=350,
                    title='Probabilidad de enfermedad',
                    xaxis=dict(range=[0, 1.15], tickformat='.0%'),
                    plot_bgcolor='white', paper_bgcolor='white',
                    margin=dict(l=80, r=80, t=60, b=40)
                )
            )),
            dbc.Alert(
                "⚠️ Este análisis es orientativo y no sustituye el diagnóstico médico profesional.",
                color='warning', style={'marginTop': '10px'}
            )
        ])
    ], style={'marginBottom': '40px'}),

    #Descargar informe
    dbc.Button(
        "📄 Descargar informe",
        id="btn-descargar-pdf",
        color="success",
        size="lg"
    ),

    dcc.Download(id="download-pdf"),
    
    # ── FOOTER ────────────────────────────────────────────────────────────────

    html.Hr(),

    html.Div(
        "TFG — Ana Arregui Beltrán | Curso 2025–2026",
        style={
            'textAlign': 'center',
            'color': '#556B2F',
            'fontSize': '12px',
            'marginBottom': '15px'
        }
    ),

], fluid=True)



In [8]:
@app.callback(
    Output("grafico-ecg","figure"),
    Input("btn-analizar","n_clicks"),
    State('input-ruta', 'value')
)
def pintar_ecg(n_clicks,value):
    
    if n_clicks is None:
        return go.Figure()

    if value is None:
        fig_vacia = go.Figure()
        fig_vacia.update_layout(
            title="No se ha cargado ningún ECG",
            height=500,
            plot_bgcolor="white",
            paper_bgcolor="white"
        )
        return fig_vacia

    try:
        BASE = Path(value)
        
        L, x, sig, spacing, BASE, n_show, lead_names, fs = leer_ecg(BASE)

    except Exception as e:

        fig_error = go.Figure()
        fig_error.update_layout(
            title=f"Error leyendo ECG: {str(e)}",
            height=500,
            plot_bgcolor="white",
            paper_bgcolor="white"
        )
        return fig_error

    fig = graficar_ecg(
        L, x, sig,
        spacing,
        BASE,
        n_show,
        lead_names,
        fs
    )

    return fig


#callback descargar informe
@app.callback(
    Output("download-pdf", "data"),
    Input("btn-descargar-pdf", "n_clicks"),

    prevent_initial_call=True
)
def descargar_pdf(n_clicks):

    # ==========================================
    # IMPORTS
    # ==========================================
    
    # ==========================================
    # NOMBRE PDF
    # ==========================================
    nombre_pdf = "informe_ecg.pdf"

    # ==========================================
    # CREAR PDF
    # ==========================================
    c = canvas.Canvas(nombre_pdf, pagesize=letter)

    # ==========================================
    # CABECERA
    # ==========================================
    c.setFont("Helvetica-Bold", 18)
    c.drawString(50, 770, "Informe Automático ECG")

    # ==========================================
    # FECHA
    # ==========================================
    fecha = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")

    c.setFont("Helvetica", 10)
    c.drawString(50, 745, f"Fecha: {fecha}")

    # ==========================================
    # RESUMEN CLÍNICO
    # ==========================================
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, 700, "Resumen clínico")

    c.setFont("Helvetica", 12)

    c.drawString(70, 670, "• Frecuencia cardíaca: 72 bpm")
    c.drawString(70, 645, "• Eje eléctrico: 45°")
    c.drawString(70, 620, "• Media RR: 0.833 s")
    c.drawString(70, 595, "• SDNN (HRV): 0.045 s")
    c.drawString(70, 570, "• Latidos detectados: 11")

    # ==========================================
    # DIAGNÓSTICO
    # ==========================================
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, 520, "Diagnóstico")

    c.setFont("Helvetica", 12)

    c.drawString(70, 490, "• NORM: 82%")
    c.drawString(70, 465, "• HYP: 60%")
    c.drawString(70, 440, "• STTC: 45%")

    # ==========================================
    # AVISO LEGAL
    # ==========================================
    c.setFont("Helvetica-Oblique", 10)

    c.drawString(
        50,
        380,
        "Este análisis es orientativo y no sustituye el diagnóstico médico profesional."
    )

    # ==========================================
    # FOOTER
    # ==========================================
    c.setFont("Helvetica", 9)

    c.drawString(
        50,
        40,
        "TFG — Ana Arregui Beltrán | Curso 2025–2026"
    )

    # ==========================================
    # GUARDAR PDF
    # ==========================================
    c.save()

    # ==========================================
    # DESCARGAR PDF
    # ==========================================
    return dcc.send_file(nombre_pdf)

In [9]:
if __name__ == "__main__":
    app.run(debug=True, port=8050)